In [2]:
from f5_tts.api import F5TTS
import torch
from datasets import load_dataset, DatasetDict, concatenate_datasets
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import os
import numpy as np
import pandas as pd
import librosa
import evaluate
import re
import unicodedata



device = "cuda" if torch.cuda.is_available() else "cpu"

# model_name = "mansi_v1_Base"
# model_name = "mansi_v2_Hun"
# model_name = "mansi_v3_Rus"
# model_name = "mansi_v4_Hun_v2"
# model_name = "mansi_v5_Rus"

# model_path = f"models/{model_name}/model.pt"
# vocab_path = f"models/{model_name}/vocab.txt"



model_name = "F5TTS_rus"
model_path = f"F5-TTS/ckpts/F5TTS_rus/model.pt"
vocab_path = f"F5-TTS/data/base_vocab/vocab.txt"


ref_audio_path = "../data/tts_dataset/val/wavs/76f24ba6-c40d-447b-9f2d-a60a6a9f9620.wav"
ref_text = "Тувыл мощ поргащлаӈкве та̄ртавет, таима̄гыс нэ̄мхуньт ат ля̄выльттавет."

model = F5TTS(ckpt_file=model_path, vocab_file=vocab_path, device=device)

pred_path = f'pred_{model_name}'

stt_model_path = "../stt/models/whisper_small_mansi_v2"

/home/dev/workspace/shidlovskiy/mansi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-28 08:50:31.078986: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/dev/workspace/shidlovskiy/mansi/.venv/lib/python3.12/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  F5-TTS/data/base_vocab/vocab.txt
token :  custom
model :  F5-TTS/ckpts/F5TTS_rus/model.pt 



In [3]:
tts_path = "../data/tts_dataset"
tts_test_dataset = load_dataset('csv', data_files=tts_path+'/test/metadata.csv', delimiter='\t', split='train')

stt_path = "../data/stt_dataset"
stt_test_dataset = load_dataset('csv', data_files=stt_path+'/test/metadata.csv', delimiter='\t', split='train')

bonus_path = "../data/bonus_dataset"
bonus_test_dataset = load_dataset('csv', data_files=bonus_path+'/test/metadata.csv', delimiter='\t', split='train')

test_dataset = DatasetDict({
    "test": concatenate_datasets([stt_test_dataset, tts_test_dataset, bonus_test_dataset]),
})

Generating train split: 1371 examples [00:00, 261964.87 examples/s]
Generating train split: 890 examples [00:00, 241754.46 examples/s]
Generating train split: 1534 examples [00:00, 286927.50 examples/s]


In [4]:
test_dataset

DatasetDict({
    test: Dataset({
        features: ['audio_file', 'text', 'speaker'],
        num_rows: 3795
    })
})

In [5]:
from pathlib import Path
os.makedirs(pred_path, exist_ok=True)

for sample in test_dataset['test']:
    file_path = Path(f'{pred_path}/{sample['audio_file']}')
    if not file_path.exists():
        print(f'{pred_path}/{sample['audio_file']}')
        model.infer(ref_file=ref_audio_path,
                ref_text=ref_text,
                gen_text=sample['text'],
                file_wave=f'{pred_path}/{sample['audio_file']}',
                speed=0.75
            )

/home/dev/workspace/shidlovskiy/mansi/.venv/lib/python3.12/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


pred_F5TTS_rus/34e4ee43-bbae-4cc7-be02-ae8480a5ac4f-1.wav
Converting audio...
Using custom reference text...

ref_text   Тувыл мощ поргащлаӈкве та̄ртавет, таима̄гыс нэ̄мхуньт ат ля̄выльттавет. 
gen_text 0 Ке̄рнялил нёхыс пувуӈкв


Generating audio in 1 batches...


100%|██████████| 1/1 [00:01<00:00,  1.97s/it]


pred_F5TTS_rus/456223c0-5d19-4020-bd15-b0caf0dcb275-1.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   Тувыл мощ поргащлаӈкве та̄ртавет, таима̄гыс нэ̄мхуньт ат ля̄выльттавет. 
gen_text 0 Тав ул ат порславем.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:00<00:00,  1.54it/s]


...

100%|██████████| 1/1 [00:00<00:00,  2.03it/s]


pred_F5TTS_rus/bb1729d2-670f-49e1-9939-f3e02fcd7908.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   Тувыл мощ поргащлаӈкве та̄ртавет, таима̄гыс нэ̄мхуньт ат ля̄выльттавет. 
gen_text 0 Ме̄н наӈын янытлаӈкв таӈхиламе̄н.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:00<00:00,  2.01it/s]


pred_F5TTS_rus/81b75d86-d021-4b98-9953-0304215798a1.wav
Converting audio...
Using cached preprocessed reference audio...
Using custom reference text...

ref_text   Тувыл мощ поргащлаӈкве та̄ртавет, таима̄гыс нэ̄мхуньт ат ля̄выльттавет. 
gen_text 0 О̄лнэ колынт ащирмаг ке о̄лы, ос ре̄г ма̄гсыл са̄в олн ойтэгы̄н, вит лю̄льсаӈ со̄схаты, кол по̄хат тӯйт ат ке хилаве, тыи ос мо̄т ва̄рмалит урыл округувт рӯпитан Жилстройнадзор колн нэ̄пак хансуӈкве э̄ри.


Generating audio in 1 batches...


100%|██████████| 1/1 [00:00<00:00,  1.30it/s]


In [6]:
def normalize(text: str) -> str:
    text = text.lower()
    
    normalized_chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat == "Zs":
            normalized_chars.append(ch)
        elif unicodedata.combining(ch) and ch in "̄":
            normalized_chars.append(ch)
        elif ch in ["ӈ"]:
            normalized_chars.append(ch)

    return re.sub(r"\s+", " ", "".join(normalized_chars)).strip()

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")


def compute_metrics(pred_text, test_text):
    orh_wer = 100 * metric_wer.compute(predictions=pred_text, references=test_text)
    orh_cer = 100 * metric_cer.compute(predictions=pred_text, references=test_text)

    pred_str_norm = [normalize(pred) for pred in pred_text]
    label_str_norm = [normalize(label) for label in test_text]

    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [ 
        label_str_norm[i] for i in range(len(label_str_norm))if len(label_str_norm[i]) > 0
    ]

    wer = 100 * metric_wer.compute(predictions=pred_str_norm, references=label_str_norm)
    cer = 100 * metric_cer.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer": wer, "cer": cer, "orh_wer": orh_wer, "orh_cer": orh_cer}

In [7]:
wer_list = []
cer_list = []
orh_wer_list = []
orh_cer_list = []

processor = WhisperProcessor.from_pretrained(stt_model_path, language="hungarian", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(stt_model_path).to(device)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config._from_model_config = True

dataset = pd.DataFrame(columns=['whisper_out', 'label', 'wer', 'cer', 'orh_wer', 'orh_cer'])

for sample in test_dataset['test']:
    pred_file, sr = librosa.load(f'{pred_path}/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    pred_ids = model.generate(inputs.input_features, max_length=225)
    pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

    metrics = compute_metrics([pred_text], [test_text])
    new_row = pd.DataFrame([{
        'audio_file': f'{pred_path}/{sample['audio_file']}',
        'whisper_out': pred_text, 
        'label': test_text, 
        'wer': metrics['wer'], 
        'cer': metrics['cer'], 
        'orh_wer': metrics['orh_wer'], 
        'orh_cer': metrics['orh_cer']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)

dataset.to_csv(f'predicts_{model_name}.csv', index=False, sep="\t")

/home/dev/workspace/shidlovskiy/mansi/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1667: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': []}. If this is not desired, please set these values explicitly.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.ge